In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import YEARS, load_policies, load_claims, match_claims, completeness

claims = load_claims()
HOME = ["buildings", "contents"]
pol = {l: load_policies(l, "train") for l in HOME}

for line in HOME:
    p = pol[line]
    m = match_claims(claims, p, line, cap=None)          # excess applies to all home perils
    exp_y = p.groupby("year").exposure_clean.sum()
    perils = sorted(m.peril.unique())
    print(f"\n{'=' * 25} {line.upper()} {'=' * 25}")

    # [1] Reporting delay and completeness
    lag = m[m.loss_year == 2021].groupby("peril").report_lag.quantile([0.5, 0.9]).unstack()
    lag.columns = ["lag_median", "lag_p90"]
    lag["compl_2023"] = [completeness(m, pr, 2023) for pr in lag.index]
    lag["compl_2024"] = [completeness(m, pr, 2024) for pr in lag.index]
    lag["cost_share"] = m.groupby("peril").net.sum() / m.net.sum()
    print("\n[1] reporting delay (2021 losses) and completeness")
    print(lag.round(3))

    # [2] Frequency (completeness-adjusted) and gross severity by peril and year
    rows = []
    for pr in perils:
        for y in YEARS:
            s = m[(m.peril == pr) & (m.loss_year == y)]
            rows.append({"peril": pr, "year": y,
                         "freq": len(s) / completeness(m, pr, y) / exp_y[y] * 1000,
                         "sev": s.incurred_to_date.mean()})
    fs = pd.DataFrame(rows)
    for col, label in [("freq", "frequency per 1,000 (adjusted)"), ("sev", "gross severity (mean)")]:
        t = fs.pivot(index="peril", columns="year", values=col)
        t["annual_%"] = ((t[2024] / t[2021]) ** (1 / 3) - 1) * 100
        t["step_23_%"] = (t[[2023, 2024]].mean(axis=1) / t[[2021, 2022]].mean(axis=1) - 1) * 100
        print(f"\n[2] {label}")
        print(t.round(2))

    # [3] Large claims
    print("\n[3] net cost percentiles")
    print(m.net.quantile([0.5, 0.9, 0.99, 0.999]).round(0).to_dict(), "| max:", round(m.net.max()))
    for cap in [100_000, 250_000, 500_000]:
        print(f"cap {cap:>7,}: {(m.net > cap).sum():>3} claims, "
              f"{np.maximum(m.net - cap, 0).sum() / m.net.sum():.1%} of cost above cap")
    print(m.nlargest(5, "net")[["peril", "status", "loss_year", "incurred_to_date", "net"]].to_string(index=False))


========================= BUILDINGS =========================

[1] reporting delay (2021 losses) and completeness
                   lag_median  lag_p90  compl_2023  compl_2024  cost_share
peril                                                                     
accidental_damage         8.0     22.7       0.998       0.965       0.013
escape_of_water          11.0     35.0       0.999       0.951       0.225
fire                      3.0     10.5       0.998       0.980       0.242
flood                     8.0     22.0       0.999       0.965       0.292
storm                     8.0     25.0       0.999       0.964       0.099
subsidence               85.0    253.6       0.993       0.688       0.128

[2] frequency per 1,000 (adjusted)
year                2021   2022   2023   2024  annual_%  step_23_%
peril                                                             
accidental_damage   6.21   6.64   6.46   6.84      3.28       3.51
escape_of_water    17.59  16.81  16.76  17.94   

In [2]:
weather = pd.read_parquet("../data/weather.parquet")
p = pol["buildings"]
m = match_claims(claims, p, "buildings", cap=None)
m["lq"] = m.loss_date.dt.to_period("Q")

# [A] Quarterly median gross severity
print("[A] quarterly median gross severity")
print(m[m.peril.isin(["escape_of_water", "flood", "storm"])]
      .groupby(["lq", "peril"]).incurred_to_date.median().unstack().round(0).to_string())

# [B] Frequency by weather quintile (policy-year weather attached to each row)
cnt = m.groupby(["row_id", "peril"]).size().unstack(fill_value=0)
w = (p[["row_id", "address_id", "year", "exposure_clean"]]
     .merge(weather, on=["address_id", "year"], how="left")
     .merge(cnt, left_on="row_id", right_index=True, how="left")
     .fillna({c: 0 for c in cnt.columns}))
print("\nrows without weather:", round(w.rainfall_mm.isna().mean(), 4))

for peril, var in [("subsidence", "dry_summer_index"), ("flood", "rainfall_mm"),
                   ("storm", "wind_exposure_index"), ("fire", "wildfire_index")]:
    w["bucket"] = pd.qcut(w[var], 5, labels=["Q1 low", "Q2", "Q3", "Q4", "Q5 high"], duplicates="drop")
    g = w.groupby("bucket", observed=True)[[peril, "exposure_clean"]].sum()
    print(f"\n[B] {peril} frequency per 1,000 by {var} quintile")
    print((g[peril] / g.exposure_clean * 1000).round(2).to_string())

print("\n[B] average weather by year (exposure-weighted)")
cols = ["dry_summer_index", "rainfall_mm", "wind_exposure_index", "wildfire_index"]
print(w.dropna(subset=cols).groupby("year")
       .apply(lambda d: pd.Series(np.average(d[cols], weights=d.exposure_clean, axis=0), cols)).round(2))

# [C] 2025 weather for test addresses
tb = load_policies("buildings", "test")
print("\n[C] weather years available:", sorted(weather.year.unique()))
print("test addresses with 2025 weather:",
      round(tb.address_id.isin(weather.loc[weather.year == 2025, "address_id"]).mean(), 4))

[A] quarterly median gross severity
peril   escape_of_water    flood   storm
lq                                      
2021Q1           3584.0  18366.0  2161.0
2021Q2           3326.0  18313.0  2191.0
2021Q3           3409.0  17746.0  2150.0
2021Q4           3302.0  18565.0  1911.0
2022Q1           3495.0  16859.0  2225.0
2022Q2           3651.0  22234.0  1763.0
2022Q3           3470.0  18982.0  2159.0
2022Q4           3734.0  22702.0  2227.0
2023Q1           3934.0  23861.0  2205.0
2023Q2           3876.0  25607.0  2253.0
2023Q3           4063.0  23529.0  2543.0
2023Q4           3569.0  19234.0  2325.0
2024Q1           4041.0  22640.0  2756.0
2024Q2           4010.0  24029.0  2780.0
2024Q3           4258.0  24208.0  2506.0
2024Q4           4559.0  23295.0  2785.0

rows without weather: 0.0

[B] subsidence frequency per 1,000 by dry_summer_index quintile
bucket
Q1 low     1.36
Q2         1.64
Q3         1.40
Q4         2.32
Q5 high    4.21

[B] flood frequency per 1,000 by rainfall_mm q

In [1]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from src.target import LINES, load_policies, load_claims, build_model_target

claims = load_claims()
summary = {}
for line in LINES:
    p = load_policies(line, "train")
    mt, comp, loading = build_model_target(claims, p, line)
    chk = mt.groupby("year")[["target", "exposure_clean", "weight"]].sum()
    chk["raw_per_exp"] = chk.target / chk.exposure_clean
    chk["adj_per_weight"] = chk.target / chk.weight
    print(f"\n===== {line} =====")
    print(pd.concat([comp.rename("completeness"), chk[["raw_per_exp", "adj_per_weight"]]], axis=1).round(3))
    level = mt.target.sum() / mt.weight.sum()
    spread = chk.adj_per_weight.max() / chk.adj_per_weight.min() - 1
    summary[line] = {"2025 level": level, "loading": loading, "level + loading": level + loading,
                     "year spread %": spread * 100}
print("\n", pd.DataFrame(summary).T.round(1))


===== motor =====
      completeness  raw_per_exp  adj_per_weight
2021         1.000      550.679         550.825
2022         0.997      546.983         548.363
2023         0.979      539.233         551.024
2024         0.816      440.626         540.015

===== buildings =====
      completeness  raw_per_exp  adj_per_weight
2021         1.000      456.429         456.429
2022         1.000      452.876         452.876
2023         0.998      459.602         460.576
2024         0.919      449.056         488.542

===== contents =====
      completeness  raw_per_exp  adj_per_weight
2021         1.000      106.088         106.088
2022         1.000      101.142         101.142
2023         0.997      108.738         109.024
2024         0.964      104.213         108.099

            2025 level  loading  level + loading  year spread %
motor           547.9     16.0            563.9            2.0
buildings       464.2      2.3            466.6            7.9
contents        106.1    

In [2]:
from src.target import adjust_claims, match_claims, completeness, ADJ

for line, focus in [("buildings", 2024), ("contents", 2022)]:
    p = load_policies(line, "train")
    m = adjust_claims(match_claims(claims, p, line, cap=None), ADJ[line])
    exp_y = p.groupby("year").exposure_clean.sum()
    rows = []
    for pr in sorted(m.peril.unique()):
        for y in [2021, 2022, 2023, 2024]:
            s = m[(m.peril == pr) & (m.loss_year == y)]
            rows.append({"peril": pr, "year": y,
                         "per_exp": s.net_adj.sum() / completeness(m, pr, y) / exp_y[y]})
    t = pd.DataFrame(rows).pivot(index="peril", columns="year", values="per_exp")
    t["others_avg"] = t.drop(columns=focus).mean(axis=1)
    t[f"diff_{focus}"] = t[focus] - t["others_avg"]
    print(f"\n===== {line}: 2025-level cost per exposure by peril (focus {focus}) =====")
    print(t.round(1))


===== buildings: 2025-level cost per exposure by peril (focus 2024) =====
year                2021   2022   2023   2024  others_avg  diff_2024
peril                                                               
accidental_damage    5.7    6.3    5.5    6.3         5.8        0.4
escape_of_water    114.1  104.4  107.9  116.7       108.8        7.9
fire                88.2   96.1   91.4  102.5        91.9       10.6
flood              144.7  138.0  135.6  134.2       139.4       -5.2
storm               46.7   46.6   53.3   48.4        48.9       -0.4
subsidence          57.0   61.6   66.9   80.4        61.8       18.6

===== contents: 2025-level cost per exposure by peril (focus 2022) =====
year               2021  2022  2023  2024  others_avg  diff_2022
peril                                                           
accidental_damage   7.8   7.2   7.6   7.9         7.8       -0.6
escape_of_water    22.0  23.4  22.6  23.4        22.7        0.8
flood              31.6  26.3  33.4  27

In [1]:
import sys
sys.path.append("..")
import numpy as np
from src.target import ADJ, load_policies, load_claims, match_claims, adjust_claims

claims = load_claims()
p = load_policies("motor", "train")
m = adjust_claims(match_claims(claims, p, "motor", cap=None), ADJ["motor"])

closed = m[m.status == "closed"]
rate = closed.groupby("peril").recoveries.sum() / closed.groupby("peril").incurred_to_date.sum()
print("[closed recovery rate by peril]\n", rate.round(3))

# Expected recoveries on open claims at the closed-claim rate, at 2025 level, capped as in the target
o = m[m.status == "open"].copy()
sev = (1 + o.peril.map(ADJ["motor"]["sev_trend"]).fillna(0)) ** (2025 - o.loss_year)
o["exp_rec"] = o.incurred_to_date * o.peril.map(rate).fillna(0) * sev
net_new = np.minimum(np.maximum(np.maximum(o.incurred_to_date * sev - o.excess, 0) - o.exp_rec, 0),
                     ADJ["motor"]["cap"])
reduction = (o.net_adj - net_new * (o.net_adj / np.minimum(np.maximum(o.incurred_to_date * sev - o.excess, 0), ADJ["motor"]["cap"]).replace(0, np.nan)).fillna(1)).sum()
print(f"\nopen claims: {len(o):,} | reduction in motor target: {reduction / m.net_adj.sum():.2%} of total")
print(o.groupby("loss_year").apply(lambda d: d.exp_rec.sum() / m.loc[m.loss_year == d.name, "net_adj"].sum()).round(4)
      .rename("reduction share by loss year"))

[closed recovery rate by peril]
 peril
bodily_injury       0.000
glass_hail          0.000
own_damage          0.236
theft               0.040
third_party_prop    0.000
dtype: float64

open claims: 7,422 | reduction in motor target: 0.34% of total
loss_year
2021    0.0000
2022    0.0000
2023    0.0001
2024    0.0160
Name: reduction share by loss year, dtype: float64
